# 🧠 Masterclass: Prompt Engineering para Científicos de Datos

¡Bienvenido/a! En la Ciencia de Datos moderna, la diferencia entre un analista promedio y un **Senior Data Scientist** no es quién memoriza más parámetros de `scikit-learn` o `pandas`, sino **quién sabe diseñar especificaciones técnicas precisas (Prompts) para que la IA escriba pipelines robustos, pruebas estadísticas rigurosas y productos de datos libres de *Data Leakage***.

> 🎯 **Cómo usar este Notebook:**
> 1. En cada sección aprenderás **cómo estructurar el prompt profesional**, comparando un *Prompt Novato ❌* contra un *Prompt Senior ✅* y desglosando **por qué funciona cada instrucción**.
> 2. Tendrás un **Generador de Prompts Dinámico en Python** que inspecciona tu `DataFrame` real y te imprime el prompt exacto listo para **copiar y pegar en ChatGPT, Claude o Gemini**.
> 3. Debajo de cada ejemplo encontrarás un **Espacio de Práctica en Blanco** para que pegues el código que te genere la IA y lo pongas a prueba con el CSV de esta clase (`datos_clientes_saas.csv`).


---
## 📐 1. Anatomía de un Prompt Profesional: El Framework C.R.E.F.O.

Cuando un Científico de Datos le habla a un LLM, **nunca** le pide *"hazme un análisis"* o *"limpia este CSV"*. Los modelos de lenguaje no pueden adivinar la distribución de tus variables ni las reglas de tu negocio. Para obtener respuestas de nivel producción a la primera, usamos la estructura **C.R.E.F.O.**:

| Letra | Bloque | ¿Qué debe incluir un Científico de Datos? | ¿Qué desastre evita? |
| :---: | :--- | :--- | :--- |
| **C** | **Contexto Técnico (Data Schema)** | Dimensiones (`shape`), nombres exactos de columnas, tipos (`dtypes`), % de nulos, cardinalidad y 3 valores de ejemplo. | Evita que el LLM invente nombres de columnas (`df['age']` cuando tu columna se llama `edad_cliente`) o asuma tipos incorrectos. |
| **R** | **Rol y Especialidad** | El perfil técnico exacto y el dominio del negocio (ej. *Staff Data Scientist experto en retención SaaS y estadística no paramétrica*). | Cambia el vocabulario y el rigor matemático del modelo (deja de usar promedios simples y empieza a hablar de medianas, IQR y tamaños del efecto). |
| **E** | **Especificación Algorítmica** | Paso a paso numerado del flujo matemático, librerías permitidas y restricciones estadísticas (ej. *imputar por mediana agrupada sin data leakage*). | Evita que la IA haga un `df.dropna()` destructivo o impute con la media global distorsionando la varianza. |
| **F** | **Formato de Salida** | Estándares de ingeniería de software: funciones modulares, `Type Hints`, `docstrings`, sin código incompleto (`# TODO`). | Asegura código limpio, reutilizable y listo para producción. |
| **O** | **Output de Verificación (`assert`)** | Pedir explícitamente pruebas de validación de datos al final del script (`assert df.isna().sum().sum() == 0`). | Te garantiza que si el código generado tiene un bug silencioso, falle inmediatamente y lo detectes. |

---
### 🧠 Técnicas Avanzadas de Prompting aplicadas a Ciencia de Datos
1. **Schema-First Prompting (Contexto basado en Metadatos):** Nunca subas un CSV confidencial de 500 MB al prompt. Extrae programáticamente los tipos de datos, el conteo de nulos y los percentiles y pásalos como texto Markdown.
2. **Chain-of-Thought (CoT) Estadístico:** Obliga al LLM a **verificar supuestos matemáticos antes de elegir una fórmula** (ej. *"Primero evalúa normalidad con Shapiro-Wilk; SI p > 0.05 usa T-Test, DE LO CONTRARIO usa Mann-Whitney U"*).
3. **Few-Shot Prompting para Reglas de Negocio:** Dale 2 o 3 ejemplos de cómo quieres transformar categorías sucias o cómo quieres que redacte un hallazgo financiero.
4. **Defensive Prompting (Prevención de *Data Leakage*):** Incluye siempre una cláusula negativa: *"PROHIBIDO calcular estadísticos globales antes de dividir en train/test o mezclar segmentos de distinta escala"*.


---
## 🛠️ Paso 0: Generemos nuestra Base de Datos CSV y el Extractor Automático de Contexto

Ejecuta las siguientes dos celdas.
1. La primera creará en tu carpeta el archivo `datos_clientes_saas.csv` (con **1,215 registros**, nulos intencionales, errores de captura y outliers extremos) para que tengas una base realista sobre la cual probar los prompts.
2. La segunda celda contiene una **herramienta fundamental de Prompt Engineering**: una función en Python que lee cualquier `DataFrame` y te redacta automáticamente el bloque `[CONTEXTO]` perfecto para tus prompts.


In [1]:
import numpy as np
import pandas as pd

# Generamos el dataset realista 'datos_clientes_saas.csv' para usar en nuestros prompts
np.random.seed(42)
n = 1200

segmentos = np.random.choice(['Enterprise', 'SMB', 'Startup', 'enterprise ', 'smb'], size=n, p=[0.20, 0.45, 0.25, 0.05, 0.05])
regiones = np.random.choice(['LATAM', 'North America', 'EMEA', 'APAC'], size=n, p=[0.40, 0.30, 0.20, 0.10])
canales = np.random.choice(['Organic Search', 'Paid Ads', 'Referral', 'Outbound Sales'], size=n)

antiguedad_meses = np.random.gamma(shape=2.5, scale=8.0, size=n).round(0)
sesiones_mes = np.random.poisson(lam=18, size=n) + (antiguedad_meses * 0.4).round(0)
tickets_soporte = np.random.poisson(lam=2.2, size=n)

base_mrr = np.where(np.char.strip(np.char.lower(segmentos)) == 'enterprise', 1200,
           np.where(np.char.strip(np.char.lower(segmentos)) == 'smb', 350, 120))
gasto_mensual_usd = np.random.normal(loc=base_mrr, scale=base_mrr * 0.25, size=n).round(2)

logit_churn = -2.2 + 0.38 * tickets_soporte - 0.06 * sesiones_mes - 0.04 * antiguedad_meses
prob_churn = 1 / (1 + np.exp(-logit_churn))
churn = np.random.binomial(1, p=np.clip(prob_churn, 0.02, 0.95), size=n)
csat_score = np.clip((8.5 - 0.6 * tickets_soporte + np.random.normal(0, 1.2, size=n)).round(1), 1.0, 10.0)

df_raw = pd.DataFrame({
    'customer_id': [f'CUST-{10000 + i}' for i in range(n)],
    'fecha_registro': pd.date_range(start='2023-01-01', periods=n, freq='12h').strftime('%Y-%m-%d'),
    'segmento': segmentos,
    'region': regiones,
    'canal_adquisicion': canales,
    'antiguedad_meses': antiguedad_meses,
    'sesiones_mes': sesiones_mes,
    'tickets_soporte': tickets_soporte,
    'csat_score': csat_score,
    'gasto_mensual_usd': gasto_mensual_usd,
    'churn': churn
})

# Inyectamos problemas reales: Nulos, Outliers negativos/extremos y Duplicados
df_raw.loc[np.random.choice([True, False], size=n, p=[0.06, 0.94]), 'gasto_mensual_usd'] = np.nan
df_raw.loc[np.random.choice([True, False], size=n, p=[0.08, 0.92]), 'csat_score'] = np.nan

outlier_idx = np.random.choice(df_raw.index, size=8, replace=False)
df_raw.loc[outlier_idx[:4], 'gasto_mensual_usd'] = [18500.0, 24000.0, -450.0, 31000.0]
df_raw.loc[outlier_idx[4:], 'antiguedad_meses'] = [240, 360, -5, 190]
df_raw = pd.concat([df_raw, df_raw.sample(15, random_state=42)], ignore_index=True)

df_raw.to_csv('datos_clientes_saas.csv', index=False)
print(f"✅ Base de datos 'datos_clientes_saas.csv' creada ({df_raw.shape[0]} filas x {df_raw.shape[1]} columnas).")
df_raw.head(4)


✅ Base de datos 'datos_clientes_saas.csv' creada (1215 filas x 11 columnas).


,customer_id,fecha_registro,segmento,region,canal_adquisicion,antiguedad_meses,sesiones_mes,tickets_soporte,csat_score,gasto_mensual_usd,churn
0,CUST-10000,2023-01-01,SMB,EMEA,Referral,33.0,27.0,0,7.7,310.19,0
1,CUST-10001,2023-01-01,smb,LATAM,Referral,26.0,22.0,3,8.3,363.67,0
2,CUST-10002,2023-01-02,Startup,LATAM,Referral,36.0,39.0,1,4.8,NaN,0
3,CUST-10003,2023-01-02,SMB,LATAM,Referral,21.0,28.0,2,6.8,310.54,0


In [2]:
# ==============================================================================
# HERRAMIENTA DE PROMPT ENGINEERING: EXTRACTOR DE METADATOS PARA EL PROMPT
# ==============================================================================
def extraer_esquema_para_prompt(df: pd.DataFrame, nombre_tabla: str = "df") -> str:
    """
    Convierte la estructura de cualquier DataFrame en un bloque de texto compacto
    listo para pegarse en la sección [CONTEXTO] de un prompt de Ciencia de Datos.
    """
    lineas = [f"DataFrame `{nombre_tabla}` con {df.shape[0]} filas y {df.shape[1]} columnas:"]
    for col in df.columns:
        dtype = str(df[col].dtype)
        nulos_pct = (df[col].isna().mean() * 100).round(1)
        n_unicos = df[col].nunique()
        if pd.api.types.is_numeric_dtype(df[col]):
            min_v, med_v, max_v = df[col].min(), df[col].median(), df[col].max()
            detalle = f"rango [{min_v} a {max_v}], mediana={med_v}"
        else:
            ejemplos = df[col].dropna().unique()[:4].tolist()
            detalle = f"ejemplos={ejemplos}"
        lineas.append(f"- `{col}` ({dtype}): {nulos_pct}% nulos | {n_unicos} valores únicos | {detalle}")
    return "\n".join(lineas)

esquema_df = extraer_esquema_para_prompt(df_raw, "df_raw")
print("📋 COPIA ESTE BLOQUE EN TUS PROMPTS PARA DAR CONTEXTO EXACTO A LA IA:\n")
print(esquema_df)


📋 COPIA ESTE BLOQUE EN TUS PROMPTS PARA DAR CONTEXTO EXACTO A LA IA:

DataFrame `df_raw` con 1215 filas y 11 columnas:
- `customer_id` (object): 0.0% nulos | 1200 valores únicos | ejemplos=['CUST-10000', 'CUST-10001', 'CUST-10002', 'CUST-10003']
- `fecha_registro` (object): 0.0% nulos | 600 valores únicos | ejemplos=['2023-01-01', '2023-01-02', '2023-01-03', '2023-01-04']
- `segmento` (object): 0.0% nulos | 5 valores únicos | ejemplos=['SMB', 'smb', 'Startup', 'Enterprise']
- `region` (object): 0.0% nulos | 4 valores únicos | ejemplos=['EMEA', 'LATAM', 'North America', 'APAC']
- `canal_adquisicion` (object): 0.0% nulos | 4 valores únicos | ejemplos=['Referral', 'Paid Ads', 'Organic Search', 'Outbound Sales']
- `antiguedad_meses` (float64): 0.0% nulos | 69 valores únicos | rango [-5.0 a 360.0], mediana=18.0
- `sesiones_mes` (float64): 0.0% nulos | 40 valores únicos | rango [11.0 a 61.0], mediana=26.0
- `tickets_soporte` (int32): 0.0% nulos | 10 valores únicos | rango [0 a 9], mediana=2.

---
## 🧹 Ejemplo 1: Prompt para Procesos de Limpieza, Imputación y Outliers (Data Wrangling)

### 1. Comparativa de Prompts

| ❌ Prompt Novato (Lo que hace todo el mundo) | ✅ Por qué falla en Ciencia de Datos |
| :--- | :--- |
| *"Tengo un CSV con clientes, hay valores nulos en `gasto_mensual_usd` y `csat_score`, duplicados y algunos datos raros. Dame el código en pandas para limpiarlo."* | 1. **Destrucción de muestra:** La IA suele aplicar `df.dropna()`, eliminando el 14% de tus clientes.<br>2. **Sesgo estadístico:** Si imputa con `df.fillna(df.mean())`, mezclará el gasto de una *Startup* (\$120/mes) con el de un cliente *Enterprise* (\$1,200/mes) y encima contaminado por outliers de \$31,000.<br>3. **Falta de trazabilidad:** Modifica el DataFrame original sin reportar qué cambió. |

---

### 2. 🎯 El Prompt Profesional (Listo para Copiar y Pegar en la IA)

```markdown
### [ROL]
Actúa como un Senior Data Scientist y Data Quality Engineer experto en preprocesamiento robusto con `pandas` y `numpy`.

### [CONTEXTO]
Tengo cargado en memoria un DataFrame llamado `df_raw` (leído desde `datos_clientes_saas.csv` con 1,215 filas y 11 columnas):
- `customer_id` (object): Identificador del cliente (existen 15 filas duplicadas).
- `fecha_registro` (object): Fecha en formato 'YYYY-MM-DD'.
- `segmento` (object): Categorías sucias con espacios y minúsculas: ['Enterprise', 'SMB', 'Startup', 'enterprise ', 'smb'].
- `region` (object): ['LATAM', 'North America', 'EMEA', 'APAC'].
- `antiguedad_meses` (float64): Rango [-5.0 a 360.0], mediana=17.0 (contiene valores negativos imposibles y outliers extremos).
- `gasto_mensual_usd` (float64): 6.0% nulos, rango [-450.0 a 31000.0], mediana=337.9 (su escala depende fuertemente de `segmento`).
- `csat_score` (float64): 8.0% nulos, rango [1.0 a 10.0], mediana=7.2.
- `sesiones_mes` (float64), `tickets_soporte` (int64), `churn` (int64: 0 o 1).

### [ESPECIFICACIÓN TÉCNICA]
Escribe una función modular `limpiar_datos_saas(df: pd.DataFrame) -> pd.DataFrame` que ejecute estrictamente en este orden:
1. Trabajar sobre una copia (`df.copy()`) para no mutar el DataFrame original.
2. Eliminar duplicados basándose en `customer_id`, conservando la primera aparición (`keep='first'`).
3. Normalizar strings en `segmento` (quitar espacios laterales, estandarizar capitalización a: `'Enterprise'`, `'SMB'`, `'Startup'`).
4. Reglas de validación física: reemplazar valores `< 0` en `antiguedad_meses` y valores `<= 0` en `gasto_mensual_usd` por `np.nan` ANTES de calcular cualquier estadístico.
5. Tratamiento de Outliers (Winsorizing Agrupado): limitar (`clip`) las columnas `gasto_mensual_usd` y `antiguedad_meses` entre el percentil 1% (P01) y 99% (P99) calculados **dentro de cada `segmento`**.
6. Imputación Jerárquica sin sesgo: imputar los `np.nan` de `gasto_mensual_usd`, `csat_score` y `antiguedad_meses` utilizando la **mediana agrupada por `['segmento', 'region']`** (usando `.groupby(...).transform('median')`).

### [FORMATO Y VERIFICACIÓN]
- Incluye Type Hints, docstring explicativo y comentarios breves del porqué estadístico de cada paso.
- Imprime en consola un resumen de auditoría (duplicados eliminados, nulos imputados por columna).
- Incluye al final un bloque de validaciones con `assert` que garantice: 0 valores nulos totales, categorías exactas en `segmento` y gastos estrictamente positivos.
- Guarda el resultado limpio en `'datos_clientes_saas_limpio.csv'`.
```

### 🔍 ¿Por qué este prompt es de nivel Senior?
- **Secuencia lógica explícita:** Si no le pides en el paso 4 convertir negativos a `NaN` *antes* de imputar, la mediana incluirá el `-450.0`.
- **Restricción de agrupación (`.groupby().transform()`):** Evita que la IA use un `SimpleImputer` global que arruinaría la relación entre el segmento y el precio del plan.


In [3]:
# 🧪 ESPACIO DE PRÁCTICA 1:
# Copia el Prompt Profesional de arriba, pégalo en tu IA favorita (ChatGPT / Claude / Gemini)
# y pega aquí abajo el código que te devuelva para ejecutarlo y verificar que pase los `assert`:

# --- PEGA AQUÍ EL CÓDIGO GENERADO POR LA IA ---


---
## 📊 Ejemplo 2: Prompt para Análisis Descriptivo y Estadística Inferencial Rigurosa

### 1. Comparativa de Prompts

| ❌ Prompt Novato | ✅ Por qué falla en Ciencia de Datos |
| :--- | :--- |
| *"Hazme un análisis descriptivo del dataframe para ver las diferencias entre los clientes que hacen churn (1) y los que no (0)."* | 1. **Promedios engañosos:** Te dará un simple `df.groupby('churn').mean()`, el cual miente cuando las distribuciones son asimétricas (como ingresos o antigüedad).<br>2. **Cero inferencia:** No te dirá si la diferencia observada es estadísticamente significativa o puro ruido aleatorio.<br>3. **Falta de Tamaño del Efecto:** Un $p$-valor pequeño en muestras grandes no significa que la variable sea útil para el negocio. |

---

### 2. 🎯 El Prompt Profesional (Con *Chain-of-Thought* Estadístico)

```markdown
### [ROL]
Actúa como un Lead Data Scientist especializado en Inferencia Estadística y Experimentación.

### [CONTEXTO]
Usando el DataFrame limpio `df_clean` (1,200 registros sin nulos), quiero comparar estadísticamente a los clientes que abandonan (`churn == 1`) frente a los clientes retenidos (`churn == 0`) en las variables numéricas:
`['tickets_soporte', 'csat_score', 'sesiones_mes', 'antiguedad_meses', 'gasto_mensual_usd']`.

### [ESPECIFICACIÓN TÉCNICA (CHAIN-OF-THOUGHT ESTADÍSTICO)]
Escribe una función `comparar_grupos_churn(df: pd.DataFrame) -> pd.DataFrame` utilizando `scipy.stats` que para cada variable numérica siga este árbol de decisión:
1. **Estadísticos Paramétricos y No Paramétricos:** Calcula para ambos grupos (`churn=0` y `churn=1`) tanto la Media ± Desviación Estándar como la Mediana y el Rango Intercuartílico (IQR = Q75 - Q25), además del coeficiente de asimetría (`skew`).
2. **Verificación de Normalidad:** Aplica el test de Shapiro-Wilk (`scipy.stats.shapiro`) a una muestra de hasta 500 observaciones de cada grupo.
3. **Selección Dinámica de Prueba de Hipótesis:**
   - SI ambos grupos tienen $p > 0.05$ en Shapiro-Wilk (distribución normal), ejecuta el **T-test de Welch** (`ttest_ind(..., equal_var=False)`).
   - DE LO CONTRARIO (no normal), ejecuta la prueba no paramétrica **U de Mann-Whitney** (`mannwhitneyu(..., alternative='two-sided')`).
4. **Corrección por Comparaciones Múltiples:** Ajusta los $p$-valores obtenidos usando el método de Bonferroni o Benjamini-Hochberg (FDR).
5. **Tamaño del Efecto (Impacto Práctico):** Calcula la **$d$ de Cohen** (en valor absoluto) y clasifícala en una columna `'Magnitud_Impacto'` como: `'Grande (>=0.8)'`, `'Mediano (>=0.5)'`, `'Pequeño (>=0.2)'` o `'Despreciable (<0.2)'`.

### [FORMATO DE SALIDA]
- Devuelve un `pd.DataFrame` resumen ordenado de mayor a menor según el Tamaño del Efecto (`Cohen_d`), listo para mostrarse en Jupyter Notebook con formato limpio.
- Agrega al final una breve función que interprete automáticamente en texto cuáles son las 2 variables con mayor impacto real en el abandono.
```

### 🔍 ¿Por qué este prompt es de nivel Senior?
- **Árbol de decisión condicional (*If/Else* matemático):** En lugar de pedirle a la IA *"aplica un T-test"*, le obligas a programar la validación del supuesto de normalidad primero.
- **Tamaño del Efecto ($d$ de Cohen) + Corrección FDR:** Así es como trabaja un científico de datos profesional para no reportar falsos positivos a dirección.


In [4]:
# 🧪 ESPACIO DE PRÁCTICA 2:
# Pega el Prompt #2 en la IA y ejecuta aquí abajo la función estadística que te genere:

# --- PEGA AQUÍ EL CÓDIGO GENERADO POR LA IA ---


---
## 🎨 Ejemplo 3: Prompt para Graficación de Nivel Ejecutivo (*Storytelling with Data*)

### 1. Comparativa de Prompts

| ❌ Prompt Novato | ✅ Por qué falla en Ciencia de Datos |
| :--- | :--- |
| *"Hazme unas gráficas en matplotlib y seaborn para ver el churn por tickets de soporte y por segmento."* | 1. **Gráficos "Arcoíris" sin foco:** Usa las paletas por defecto donde todas las barras llaman la atención por igual.<br>2. **Títulos descriptivos aburridos:** Pone títulos como *"Churn vs Tickets"* en vez de comunicar el hallazgo (*"A partir del 3er ticket, el Churn se duplica"*).<br>3. **Ruido visual (Chartjunk):** Deja bordes innecesarios, sin etiquetas de datos directas ni líneas de referencia del promedio global. |

---

### 2. 🎯 El Prompt Profesional (Diseño Editorial y Jerarquía Visual)

```markdown
### [ROL]
Actúa como un Especialista en Visualización de Datos y Storytelling Ejecutivo (estilo Cole Nussbaumer Knaflic en "Storytelling with Data"), experto en `matplotlib` y `seaborn`.

### [CONTEXTO]
Queremos presentar al Comité Directivo una lámina visual de alta calidad basada en `df_clean` que demuestre cómo la fricción en soporte (`tickets_soporte`) y la baja frecuencia de uso (`sesiones_mes`) disparan el abandono (`churn`).

### [ESPECIFICACIÓN VISUAL Y DE CÓDIGO]
Escribe un bloque de código en Python que genere una figura de 2 paneles (`figsize=(15, 6)`, `dpi=120`):

1. **Panel Izquierdo (Gráfico de Barras con Acento Focal):**
   - Calcula la tasa de `churn` (%) para cada número de `tickets_soporte` (agrupando 5 o más tickets en la categoría `'5+'`).
   - **Regla de Color Semántico:** Pinta en gris azulado suave (`#A8DADC`) las barras cuya tasa de churn sea menor o igual al promedio global, y resalta en rojo coral (`#E63946`) las barras que superen el promedio global de churn.
   - Traza una línea horizontal punteada (`linestyle='--'`) marcando la **Tasa de Churn Promedio Global** con su etiqueta.
   - Coloca el porcentaje exacto (`f"{val:.1f}%"`) encima de cada barra en negrita.

2. **Panel Derecho (Distribución Comparativa por Cohorte):**
   - Crea un gráfico de cajas (`sns.boxplot`) o violín que compare `sesiones_mes` (eje Y) a través de cada `segmento` (`['Enterprise', 'SMB', 'Startup']` en eje X), separado por `churn` (`hue='churn'`).
   - Usa la paleta corporativa `{0: '#457B9D', 1: '#E63946'}` y renombra la leyenda a `'Retenido (0)'` y `'Churn (1)'`.

3. **Estética Editorial (Decluttering):**
   - Elimina los bordes superior y derecho con `sns.despine(top=True, right=True)`.
   - Agrega un `fig.suptitle()` en la parte superior izquierda (`ha='left'`) que **declare la conclusión de negocio en una frase activa**, no una descripción genérica.
   - Agrega un pie de figura (`fig.text`) indicando la fuente y tamaño de la muestra ($N = 1,200$).
```

### 🔍 ¿Por qué este prompt es de nivel Senior?
- **Codificación condicional del color:** Le enseñas a la IA a usar color **estratégicamente** (gris para contexto, rojo para alerta cuando supera el umbral promedio).
- **Línea base (*Benchmark*) obligatoria:** En ciencia de datos, un número aislado no significa nada sin una línea de referencia que muestre el promedio global.


In [5]:
# 🧪 ESPACIO DE PRÁCTICA 3:
# Pega el Prompt #3 en la IA y ejecuta aquí el código de visualización que te genere:

# --- PEGA AQUÍ EL CÓDIGO GENERADO POR LA IA ---


---
## ⚙️ Ejemplo 4: Prompt para Feature Engineering Basado en Hipótesis de Dominio

### 1. Comparativa de Prompts

| ❌ Prompt Novato | ✅ Por qué falla en Ciencia de Datos |
| :--- | :--- |
| *"Crea nuevas variables en mi dataframe para mejorar la predicción de churn."* | 1. **Variables caja negra sin sentido:** Multiplica o eleva al cuadrado columnas al azar sin ninguna hipótesis de negocio que puedas explicar al stakeholder.<br>2. **Errores de división por cero (`inf`):** Divide entre `sesiones_mes` o `tickets_soporte` sin considerar que pueden valer `0`, llenando el DataFrame de `np.inf`.<br>3. **No mide si aportan valor:** Crea columnas pero no comprueba si tienen mayor ganancia de información (*Mutual Information*) que las originales. |

---

### 2. 🎯 El Prompt Profesional (Orientado a Hipótesis + Evaluación con *Mutual Information*)

```markdown
### [ROL]
Actúa como un Staff Machine Learning Engineer y Kaggle Grandmaster experto en modelos tabulares de retención SaaS.

### [CONTEXTO]
Tenemos el DataFrame `df_clean` con las variables originales:
`['segmento', 'region', 'canal_adquisicion', 'antiguedad_meses', 'sesiones_mes', 'tickets_soporte', 'csat_score', 'gasto_mensual_usd']` y el target binario `churn` (0 o 1).

### [ESPECIFICACIÓN TÉCNICA]
Escribe una función `generar_features_saas(df: pd.DataFrame) -> pd.DataFrame` que construya 5 nuevas variables basadas en **hipótesis de comportamiento del usuario**:
1. `tasa_friccion_soporte`: Razón entre `tickets_soporte` y `sesiones_mes` (¿qué proporción de las visitas del cliente terminan en un problema técnico?).
2. `costo_por_sesion_usd`: Razón entre `gasto_mensual_usd` y `sesiones_mes` (mide el valor percibido; un cliente que paga mucho y usa poco el software está en alto riesgo de cancelar).
3. `customer_health_score`: Índice compuesto que premia satisfacción (`csat_score`) y uso logarítmico (`np.log1p(sesiones_mes)`), pero penaliza el volumen de quejas (`tickets_soporte`).
4. `ltv_acumulado_usd`: Estimación del ingreso histórico acumulado (`gasto_mensual_usd * antiguedad_meses`).
5. `esfuerzo_precio_relativo`: Razón entre el `gasto_mensual_usd` del cliente y la mediana de gasto de su mismo `segmento` (usando `.groupby('segmento').transform('median')`).

### [RESTRICCIONES Y VALIDACIÓN MATEMÁTICA]
- **Protección numérica:** Suma una constante de suavizado ($\epsilon = 1.0$) en todos los denominadores que puedan tomar valor cero para evitar `ZeroDivisionError` o `np.inf`.
- **Evaluación de Ganancia de Información:** Incluye una segunda función `evaluar_importancia_features(df_fe)` que calcule `mutual_info_classif` (de `sklearn.feature_selection`) y la correlación de Pearson con `churn` tanto para las 5 variables originales como para las 5 nuevas, devolviendo un DataFrame comparativo ordenado por *Mutual Information*.
```

### 🔍 ¿Por qué este prompt es de nivel Senior?
- **Fórmulas ancladas a psicología del consumidor:** Le das a la IA el concepto de negocio (*Fricción*, *Costo por Uso*, *Salud de Cuenta*) y la IA se encarga de traducirlo a operaciones vectorizadas en `numpy`/`pandas`.
- **Cláusula anti-`inf` ($\epsilon = 1$):** Una regla clásica que separa el código académico del código de producción.


In [6]:
# 🧪 ESPACIO DE PRÁCTICA 4:
# Pega el Prompt #4 en la IA y prueba aquí abajo tus nuevas variables y su ranking de Mutual Information:

# --- PEGA AQUÍ EL CÓDIGO GENERADO POR LA IA ---


---
## 💡 Ejemplo 5: Prompt para Descubrimiento Automático de Insights y Cuantificación Financiera ($ ARR)

### 1. Comparativa de Prompts

| ❌ Prompt Novato | ✅ Por qué falla en Ciencia de Datos |
| :--- | :--- |
| *"Analiza mis datos de churn y dime qué insights importantes encuentras y qué recomiendas hacer."* | 1. **Consejos genéricos de libro:** Te responderá cosas obvias como *"Mejora el soporte técnico y ofrece descuentos"*.<br>2. **Habla en métricas técnicas, no en dinero:** Ningún Director General (CEO) o VP de Ventas toma decisiones porque *"la correlación es 0.42"*; toman decisiones cuando les dices **"hay \$380,000 USD anuales (ARR) concentrados en 140 cuentas críticas y podemos salvar \$95,000 USD con esta regla"**. |

---

### 2. 🎯 El Prompt Profesional (Micro-Segmentación + Impacto en ARR + Regla de Pareto 80/20)

```markdown
### [ROL]
Actúa como un Director de Ciencia de Datos y Estrategia Financiera reportando al Comité Ejecutivo (CEO, CFO y VP de Customer Success).

### [CONTEXTO]
Contamos con el DataFrame enriquecido `df_fe` que incluye `gasto_mensual_usd`, `churn`, `segmento`, `region`, `tickets_soporte`, `sesiones_mes`, `csat_score`, `tasa_friccion_soporte` y `customer_health_score`.
En nuestro modelo de negocio SaaS:
- **MRR (Monthly Recurring Revenue)** = `gasto_mensual_usd`
- **ARR (Annual Recurring Revenue)** = `gasto_mensual_usd * 12`

### [ESPECIFICACIÓN TÉCNICA]
Escribe un script en Python que realice un **Proceso de Descubrimiento de Insights Accionables**:
1. **Segmentación por Reglas de Alerta Temprana (Early Warning Tiers):**
   Clasifica a cada cliente en una nueva columna `'tier_riesgo'` con 3 niveles:
   - `'1. Crítico 🚨'`: si `customer_health_score < 8.0` O `tasa_friccion_soporte >= 0.18`.
   - `'2. Moderado ⚠️'`: si no es crítico pero `customer_health_score` está entre `8.0` y `14.0`.
   - `'3. Saludable ✅'`: el resto de los clientes.
2. **Cuantificación Financiera por Cohorte (` segmentación x tier_riesgo`):**
   Genera una tabla resumen que calcule para cada nivel de riesgo:
   - Total de clientes y % de la base total.
   - Tasa de Churn Real (%).
   - MRR Total Activo (\$ USD).
   - **ARR Anual en Riesgo (\$ USD)** (definido como el ARR de los clientes que efectivamente caen en churn dentro de ese grupo) y qué **% del ARR total perdido de la empresa** se concentra en ese nivel (Principio de Pareto 80/20).
3. **Generador Automático de Executive Briefing + Simulador de ROI:**
   Programa una función que imprima en formato Markdown un reporte ejecutivo automático respondiendo con los números exactos del DataFrame:
   - ¿Qué porcentaje de toda la pérdida financiera anual de la empresa está concentrada en el tier `'1. Crítico 🚨'`?
   - ¿Cuál es el micro-segmento (`segmento` + `region`) con mayor dinero en riesgo?
   - **Simulación de ROI:** Si el equipo de Customer Success contacta preventivamente a las cuentas críticas y logra reducir su tasa de abandono en apenas un **25%**, ¿cuántos dólares anuales (**\$ USD en ARR**) recuperará la compañía?
```

### 🔍 ¿Por qué este prompt es de nivel Senior?
- **Obliga a cruzar Probabilidad $\times$ Valor Económico:** Un cliente *Startup* con 90% de probabilidad de churn representa \$1,440 USD/año; un cliente *Enterprise* con 50% de probabilidad representa \$14,400 USD/año. Este prompt fuerza a la IA a priorizar por **valor esperado en riesgo**.


In [7]:
# 🧪 ESPACIO DE PRÁCTICA 5:
# Pega el Prompt #5 en la IA y ejecuta aquí el motor de descubrimiento de insights financieros:

# --- PEGA AQUÍ EL CÓDIGO GENERADO POR LA IA ---


---
## 🚀 Ejemplo 6: Prompt para Crear un Dashboard Interactivo en Streamlit desde un CSV

Un Científico de Datos moderno no se queda solo en el notebook: **usa Prompt Engineering para transformar su análisis en una aplicación web interactiva con Streamlit en 2 minutos**.

### 1. Comparativa de Prompts

| ❌ Prompt Novato | ✅ Por qué falla en Ciencia de Datos |
| :--- | :--- |
| *"Hazme un dashboard en Streamlit para visualizar mi archivo CSV de clientes."* | 1. **Lento y sin caché:** No usa `@st.cache_data`, recargando y recalculando todo el CSV cada vez que mueves un botón.<br>2. **Estático y sin simulador:** Solo muestra 2 gráficas simples sin filtros cruzados, sin KPIs financieros ni opción de descargar las cuentas en riesgo. |

---

### 2. 🎯 El Prompt Profesional para Generar un Dashboard de Producción en Streamlit

Copia este prompt cuando quieras que la IA te construya una app completa en Streamlit para cualquier CSV:

```markdown
### [ROL]
Actúa como un Senior Full-Stack Data Scientist especialista en crear productos de datos ejecutivos con `Streamlit` y `Plotly Express`.

### [CONTEXTO]
Quiero construir un dashboard interactivo en un único archivo llamado `dashboard_saas.py` que consuma el archivo `'datos_clientes_saas.csv'` (y que también permita subir cualquier archivo CSV equivalente desde el sidebar).
Las columnas de entrada son: `customer_id`, `fecha_registro`, `segmento`, `region`, `canal_adquisicion`, `antiguedad_meses`, `sesiones_mes`, `tickets_soporte`, `csat_score`, `gasto_mensual_usd`, `churn`.

### [ARQUITECTURA Y ESPECIFICACIÓN FUNCIONAL]
1. **Configuración y Rendimiento:**
   - Usa `st.set_page_config(page_title="...", layout="wide")`.
   - Crea una función decorada con `@st.cache_data` que cargue el CSV, ejecute automáticamente el pipeline de limpieza (duplicados, normalización de `segmento`, winsorizing P01-P99 e imputación por mediana agrupada) y calcule las variables `tasa_friccion_soporte`, `customer_health_score` y `nivel_riesgo` (`'Crítico 🚨'`, `'Moderado ⚠️'`, `'Saludable ✅'`).
2. **Barra Lateral Interactiva (`st.sidebar`):**
   - `st.file_uploader` opcional para cargar un CSV propio (con fallback automático a `'datos_clientes_saas.csv'`).
   - Filtros `multiselect` para `segmento`, `region` y `canal_adquisicion`, más un `slider` de rango para `antiguedad_meses`.
3. **Fila Superior de KPIs Ejecutivos (`st.columns(4)` con `st.metric`):**
   - Total de Clientes Filtrados, Tasa de Churn (%), MRR Mensual Activo (\$ USD) y ARR Anual en Riesgo (\$ USD).
4. **Diseño en 3 Pestañas (`st.tabs`):**
   - **Tab 1 - Diagnóstico Ejecutivo:** Gráfico de barras agrupadas de Tasa de Churn (%) por `segmento` y `region`, y gráfico de línea del impacto de `tickets_soporte` en la probabilidad de abandono.
   - **Tab 2 - Explorador Multivariado:** Scatter plot interactivo en Plotly (`sesiones_mes` vs `gasto_mensual_usd`, coloreado por `churn`, tamaño por `tickets_soporte` y `hover_data=['customer_id', 'csat_score']`) + Boxplot del `customer_health_score` por nivel de riesgo.
   - **Tab 3 - Simulador de Retención & Acción Comercial:** Un `st.slider` donde el director elija *"¿Qué % de cuentas en Riesgo Crítico logramos salvar?"* y muestre en vivo el ahorro anual proyectado en dólares (ARR), seguido de una tabla interactiva con el Top 50 de clientes en Riesgo Crítico ordenados por mayor gasto mensual y un botón `st.download_button` para exportar esa lista de llamadas en CSV.

### [FORMATO DE SALIDA]
- Entrega el código completo en un solo bloque de Python, sin fragmentos omitidos, manejando posibles divisiones por cero o filtros vacíos con mensajes amigables (`st.warning`).
```


---
### 🖥️ Bonus Incluido: ¡Ya tienes el archivo `dashboard_saas.py` y el CSV listos en tu carpeta!

Para que puedas ver en vivo cómo luce el dashboard que produce este prompt (y también puedas modificarlo o reemplazarlo con el tuyo), ya dejé guardado el archivo **`dashboard_saas.py`** junto con **`datos_clientes_saas.csv`** en tu carpeta `c:\Users\Jaime\Documents\proyectos`.

Ejecuta la siguiente celda si quieres inspeccionar o volver a generar el archivo `dashboard_saas.py`, y corre en tu terminal:
```powershell
streamlit run dashboard_saas.py
```


In [8]:
# Verificamos que tanto el CSV como el script de Streamlit estén listos en tu carpeta de proyectos
from pathlib import Path

archivos = ["datos_clientes_saas.csv", "dashboard_saas.py"]
for nombre in archivos:
    ruta = Path(nombre)
    print(f"{'✅' if ruta.exists() else '❌'} {nombre} -> {ruta.resolve()} ({ruta.stat().st_size:,} bytes)")


✅ datos_clientes_saas.csv -> C:\Users\Jaime\Documents\proyectos\datos_clientes_saas.csv (90,831 bytes)
✅ dashboard_saas.py -> C:\Users\Jaime\Documents\proyectos\dashboard_saas.py (7,710 bytes)


---
## 🧰 Anexo: Plantilla Universal (Copy-Paste Template) para tus Propios Proyectos

Cada vez que inicies un nuevo proyecto de Ciencia de Datos, copia esta plantilla maestra y rellena los corchetes:

```markdown
### [ROL]
Actúa como un [Senior Data Scientist / ML Engineer / Bioestadístico] especializado en [Dominio: Finanzas / Retail / Salud / Churn SaaS].

### [CONTEXTO DEL DATASET]
[Pega aquí el output de la función `extraer_esquema_para_prompt(df)` de la Celda 2]
- Variable objetivo (Target): `[nombre_columna_target]` ([Binaria / Continua / Multiclase]).
- Unidad de análisis: Cada fila representa un `[cliente / transacción / paciente / sensor]`.

### [OBJETIVO Y ESPECIFICACIÓN TÉCNICA]
Necesito una función en Python llamada `[nombre_funcion](df: pd.DataFrame) -> pd.DataFrame` que realice:
1. [Paso 1: Regla de limpieza / filtro / validación de supuestos]
2. [Paso 2: Transformación matemática / prueba estadística / modelo]
3. [Paso 3: Métrica de evaluación técnica y traducción a impacto de negocio ($)]

### [RESTRICCIONES DE INGENIERÍA (DEFENSIVE PROMPTING)]
- PROHIBIDO causar Data Leakage (cualquier estadístico de imputación o escalado debe calcularse por grupo o dentro de train).
- Protege las divisiones contra ceros sumando `1e-6` o `1.0` según corresponda.
- No uses funciones deprecadas de `pandas` ni modifiques el DataFrame de entrada in-place.
- Incluye Type Hints, docstring y un bloque final de `assert` para verificar la integridad del output.
```
